# FreshRetailNet — Foundation Model Forecasting with MMF (Serverless GPU)

This notebook runs **foundation time series models** on the FreshRetailNet demand data
using the [Many Model Forecasting (MMF)](https://github.com/databricks-industry-solutions/many-model-forecasting) framework
on **serverless GPU** compute.

### Prerequisites
- Run the `fresh_retail_net_mmf` notebook first to create `mmf.fresh_retail_net.demand_train` and `mmf.fresh_retail_net.demand_score`
- Attach this notebook to **Serverless** compute with **A10 GPU** accelerator and **Environment version 5+**

### Foundation Models
We run the following pre-trained foundation models (no training required — zero-shot forecasting):
- **Chronos-Bolt** (Tiny, Mini, Small, Base) — Amazon's efficient time series foundation model
- **Chronos 2** (Base, Small, Synth) — Amazon's next-gen foundation model
- **TimesFM 2.5** — Google's 200M parameter time series foundation model

## 1. Install Dependencies

In [0]:
%pip install "mmf_sa[foundation] @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git" hf_transfer --quiet
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-serverless-gpu 0.5.11+patch5 requires databricks-connect<16,>=15.4.2, but you have databricks-connect 17.3.8 which is incompatible.
databricks-serverless-gpu 0.5.11+patch5 requires mlflow<3.0,>=2.17, but you have mlflow 3.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 2. Configuration

In [0]:
import logging
import uuid

# Suppress noisy loggers on serverless
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

catalog = "mmf"
db = "fresh_retail_net"
user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]

# Verify tables exist
train_count = spark.table(f"{catalog}.{db}.demand_train").count()
score_count = spark.table(f"{catalog}.{db}.demand_score").count()
print(f"Train rows: {train_count:,}  |  Score rows: {score_count:,}")

Train rows: 90,000  |  Score rows: 7,000


## 3. Define Foundation Models

In [0]:
active_models = [
    "ChronosBoltBase",
    "Chronos2",
    "TimesFM_2_5_200m",
]

print(f"Models to run: {len(active_models)}")
for m in active_models:
    print(f"  - {m}")

Models to run: 3
  - ChronosBoltBase
  - Chronos2
  - TimesFM_2_5_200m


## 4. Run Foundation Model Forecasts

On serverless GPU, we run **one model at a time** in a loop because Spark Connect Python workers
are CPU-only even when the driver has a GPU. The `serverless=True` flag enables driver-only
prediction path.

In [0]:
from mmf_sa import run_forecast

run_id = str(uuid.uuid4())
prediction_length = 7  # 7-day forecast horizon (matches eval split)

for model in active_models:
    print(f"\n{'='*60}")
    print(f"Running {model} (run_id={run_id})")
    print(f"{'='*60}")
    run_forecast(
        spark=spark,
        train_data=f"{catalog}.{db}.demand_train",
        scoring_data=f"{catalog}.{db}.demand_score",
        scoring_output=f"{catalog}.{db}.scoring_output",
        evaluation_output=f"{catalog}.{db}.evaluation_output",
        model_output=f"{catalog}.{db}",
        group_id="unique_id",
        date_col="ds",
        target="y",
        freq="D",
        prediction_length=prediction_length,
        backtest_length=21,
        stride=7,
        metric="smape",
        train_predict_ratio=2,
        data_quality_check=False,
        resample=False,
        active_models=[model],
        dynamic_future_numerical = ["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"], 
        dynamic_future_categorical =["holiday_flag", "activity_flag"],
        experiment_path=f"/Users/{user}/mmf/fresh_retail_net",
        use_case_name="fresh_retail_net",
        run_id=run_id,
        accelerator="gpu",
        serverless=True,
    )


Running ChronosBoltBase (run_id=772b28cf-b3e3-4001-93ec-3b7ce17db099)


2026/06/11 19:53:35 INFO mlflow.tracking.fluent: Experiment with name '/Users/visvenky@amazon.com/mmf/fresh_retail_net' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


Run quality checks
Finished quality checks
Starting evaluate_score
Starting evaluate_models
Started evaluating ChronosBoltBase


/local_disk0/.ephemeral_nfs/envs/pythonEnv-65577a83-829a-4632-bebf-1e3fb1d74cff/lib/python3.12/site-packages/mlflow/system_metrics/metrics/gpu_monitor.py:11: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
2026/06/11 19:53:39 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-65577a83-829a-4632-bebf-1e3fb1d74cff/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:156: FutureWarning: Model's `predict` method contains invalid parameters: {'input_data'}. Only the following parameter names are allowed: context, model_input, and params. Note that invalid parameters will no longer be permitted in future versions.
  param_names = _check_func_signature(func, "predict")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-65577a83-829a-4

config.json: 0.00B [00:00, ?B/s]

...

2026/06/11 19:58:12 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/06/11 19:58:12 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Finished evaluating TimesFM_2_5_200m
Finished evaluate_models
Starting run_scoring
Started scoring with TimesFM_2_5_200m
Running scoring for TimesFM_2_5_200m...


INFO:mmf_sa.data_quality_checks:External regressors detected, checking resampling configuration
INFO:mmf_sa.data_quality_checks:External regressors configuration validation passed


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

Downloaded.
Finished scoring with TimesFM_2_5_200m
Finished run_scoring
Finished evaluate_score


## 5. Review Evaluation Results

In [0]:
%sql
SELECT
  model,
  COUNT(DISTINCT unique_id) AS num_series,
  ROUND(AVG(metric_value), 4) AS avg_smape,
  ROUND(PERCENTILE(metric_value, 0.5), 4) AS median_smape
FROM mmf.fresh_retail_net.evaluation_output
GROUP BY model
ORDER BY avg_smape

model,num_series,avg_smape,median_smape
TimesFM_2_5_200m,1000,0.4538,0.3812
ChronosBoltBase,1000,0.469,0.394
Chronos2,1000,0.4717,0.3988


## 6. Review Scoring Output (Future Predictions)

In [0]:
%sql
SELECT
  model,
  COUNT(DISTINCT unique_id) AS num_series,
  COUNT(*) AS total_predictions,
  MIN(ds) AS min_date,
  MAX(ds) AS max_date
FROM mmf.fresh_retail_net.scoring_output
GROUP BY model
ORDER BY model

model,num_series,total_predictions,min_date,max_date
Chronos2,1000,1000,"List(2024-06-26T00:00:00.000Z, 2024-06-27T00:00:00.000Z, 2024-06-28T00:00:00.000Z, 2024-06-29T00:00:00.000Z, 2024-06-30T00:00:00.000Z, 2024-07-01T00:00:00.000Z, 2024-07-02T00:00:00.000Z)","List(2024-06-26T00:00:00.000Z, 2024-06-27T00:00:00.000Z, 2024-06-28T00:00:00.000Z, 2024-06-29T00:00:00.000Z, 2024-06-30T00:00:00.000Z, 2024-07-01T00:00:00.000Z, 2024-07-02T00:00:00.000Z)"
ChronosBoltBase,1000,1000,"List(2024-07-03T00:00:00.000Z, 2024-07-04T00:00:00.000Z, 2024-07-05T00:00:00.000Z, 2024-07-06T00:00:00.000Z, 2024-07-07T00:00:00.000Z, 2024-07-08T00:00:00.000Z, 2024-07-09T00:00:00.000Z)","List(2024-07-03T00:00:00.000Z, 2024-07-04T00:00:00.000Z, 2024-07-05T00:00:00.000Z, 2024-07-06T00:00:00.000Z, 2024-07-07T00:00:00.000Z, 2024-07-08T00:00:00.000Z, 2024-07-09T00:00:00.000Z)"
TimesFM_2_5_200m,1000,1000,"List(2024-06-26T00:00:00.000Z, 2024-06-27T00:00:00.000Z, 2024-06-28T00:00:00.000Z, 2024-06-29T00:00:00.000Z, 2024-06-30T00:00:00.000Z, 2024-07-01T00:00:00.000Z, 2024-07-02T00:00:00.000Z)","List(2024-06-26T00:00:00.000Z, 2024-06-27T00:00:00.000Z, 2024-06-28T00:00:00.000Z, 2024-06-29T00:00:00.000Z, 2024-06-30T00:00:00.000Z, 2024-07-01T00:00:00.000Z, 2024-07-02T00:00:00.000Z)"


In [0]:
%sql
-- Sample predictions for the top-demand series across all models
WITH top_series AS (
  SELECT unique_id
  FROM mmf.fresh_retail_net.demand_train
  GROUP BY unique_id
  ORDER BY SUM(y) DESC
  LIMIT 1
)
SELECT s.model, s.ds, s.unique_id, s.y AS predicted
FROM mmf.fresh_retail_net.scoring_output s
JOIN top_series t ON s.unique_id = t.unique_id
ORDER BY s.model, s.ds

model,ds,unique_id,predicted
Chronos2,"List(2024-06-26T00:00:00.000Z, 2024-06-27T00:00:00.000Z, 2024-06-28T00:00:00.000Z, 2024-06-29T00:00:00.000Z, 2024-06-30T00:00:00.000Z, 2024-07-01T00:00:00.000Z, 2024-07-02T00:00:00.000Z)",496_267,"List(33.75, 35.5, 35.5, 36.75, 37.25, 36.75, 37.75)"
ChronosBoltBase,"List(2024-07-03T00:00:00.000Z, 2024-07-04T00:00:00.000Z, 2024-07-05T00:00:00.000Z, 2024-07-06T00:00:00.000Z, 2024-07-07T00:00:00.000Z, 2024-07-08T00:00:00.000Z, 2024-07-09T00:00:00.000Z)",496_267,"List(30.125, 29.25, 29.125, 30.0, 30.375, 31.0, 30.625)"
TimesFM_2_5_200m,"List(2024-06-26T00:00:00.000Z, 2024-06-27T00:00:00.000Z, 2024-06-28T00:00:00.000Z, 2024-06-29T00:00:00.000Z, 2024-06-30T00:00:00.000Z, 2024-07-01T00:00:00.000Z, 2024-07-02T00:00:00.000Z)",496_267,"List(31.817570912397116, 31.92422286904795, 31.16031617875405, 35.18978077201951, 36.086477674362655, 32.43499654390321, 32.33874661988753)"
